In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In this file, the processed MSW and MSW incinerated historic data for EU is used to estimate historic ash generation quantities. The results are stored according to FutuRaM data template which has the following format: 

The column names: <br> <br>
__'Waste Stream'__: SLASH <br>
__'Country'__: Either a EU24+7 country or 'EU24+7' <br>
__'Year'__: 2010-2050 (in this case, 2010-2022) <br>
__'Scenario'__: OBS/BAU/REC/CIR <br>
__'Substance main parent'__: '19 01 11*', '19 01 13*' - waste codes for MSW <br>
__'Stock/Flow ID'__:'SLASH_bottomAshesWasteInc', 'SLASH_flyAshesWasteInc' <br>
__'Value'__: Value of ash quantity in tonnes <br>
__'Unit'__: tonnes <br>
__'Data Quality'__: 2, (out of 1-4) <br>
__'Reference'__: Data sources <br>
__'Remark 1'__: Notes on missing data, interpolation <br>
__'Remark 2'__:  <br>
__'Remark 3'__: Author <br>

In [2]:
#getting BA/FA for all historic years required for the data structure
eu_msw = pd.read_csv("../../data/processed/EU_MSW_Cleaned_Data.csv")
eu_msw_his = eu_msw.loc[eu_msw["TIME_PERIOD"].isin(np.arange(2010,2022))]
eu_msw_his= eu_msw_his[['GEN','DSP_I_RCV_E','TIME_PERIOD','unit','Country']]
eu_msw_his = eu_msw_his.rename(columns = {'GEN':"MSW_Gen", "DSP_I_RCV_E":"MSW_WasteInc"})  


__Values of Ash Generation per unit waste generated__
<br>
From BAT document:<br>
    Bottom ash = 150-350 kg/t -> 15-35 % wt/wt<br>
    Boiler ash = 20-40 kg/t -> 2-4% wt/wt<br>
    Fly Ash = 15-60 -> 1.5-6 % wt/wt<br>
    <br>
    <br>
    CODES:<br>
        SLASH_flyAshesWasteInc<br>
        SLASH_bottomAshesWasteInc<br>

In [3]:
#Estimating ash quantities
eu_msw_his["SLASH_bottomAshesWasteInc"] = eu_msw_his['MSW_WasteInc']*0.28
eu_msw_his["SLASH_flyAshesWasteInc"] = eu_msw_his['MSW_WasteInc']*0.03

eu_msw_his.drop(['MSW_Gen','MSW_WasteInc'], axis = 'columns', inplace = True)

In [4]:
# creating output data structure
columns = ['Waste Stream', 'Country', 'Year', 'Scenario', 'Substance main parent',
           'Stock/Flow ID', 'Value', 'Unit', 'Data Quality', 'Reference', 'Remark 1',
           'Remark 2', 'Remark 3']
eu_msw_his= pd.melt(eu_msw_his, id_vars=['Country','unit','TIME_PERIOD'], value_vars = ['SLASH_bottomAshesWasteInc',
                                'SLASH_flyAshesWasteInc'],var_name='Stock/Flow ID', value_name='Value')

In [5]:
#Changing all values to tonnes
eu_msw_his["Value"] *=1000
eu_msw_his['unit'] = 't'
eu_msw_his = eu_msw_his.rename(columns = {'unit':"Unit"})
#Adding year, waste stream
eu_msw_his = eu_msw_his.rename(columns = {'TIME_PERIOD':'Year'})
eu_msw_his['Waste Stream'] = 'SLASH'
#Adding LowKey codes
subs_main_parent = {'SLASH_flyAshesWasteInc':'19 01 13','SLASH_bottomAshesWasteInc':'19 01 11',
                    'SLASH_boilerAshesWasteInc':'19 01 15'}
eu_msw_his['Substance main parent'] = eu_msw_his['Stock/Flow ID'].map(subs_main_parent)
#Rearrange columns

In [6]:
eu_msw_his[['Scenario']] = 'OBS'
eu_msw_his[['Reference']] = np.nan
eu_msw_his[['Remark 2']] = np.nan
eu_msw_his.loc[eu_msw_his["Country"]=='GBR',['Remark 1']] = 'Estimated from OECD MSW incineration quantities'
conditions = (((eu_msw_his["Country"]=='DNK')&(eu_msw_his["Year"]==2010)) | ((eu_msw_his["Country"]=='IRL')&(eu_msw_his["Year"]==2013)) | ((eu_msw_his["Country"]=='IRL')&(eu_msw_his["Year"]==2015)) | ((eu_msw_his["Country"]=='ISL')&(eu_msw_his["Year"]==2019)))
eu_msw_his.loc[conditions,['Remark 1']]='Missing data, estimated from Eurostat MSW incineration quantities of neighbouring years '
eu_msw_his.loc[(eu_msw_his["Country"]!='GBR')& ~conditions,['Remark 1']]='Estimated from Eurostat MSW incineration quantities'
eu_msw_his[['Remark 3']] = 'Sowmya Ravisandiran'
eu_msw_his[['Data Quality']] = 2
eu_msw_his = eu_msw_his[columns]
eu_msw_his.drop(eu_msw_his.loc[eu_msw_his["Country"]=='EU27_2020'].index, inplace = True)
#eu_msw_his.to_csv('Data_Structure_Task4.1_Task4.2.csv')